# Gym Salesman Dataset — Synthetic Data Generation

This notebook generates 12,000 synthetic gym-membership sales conversations, each labelled
`SUCCESS` or `FAILURE`. It documents the full pipeline end to end.

## Design principle

> **The label is an OUTPUT of the simulation, never an INPUT to it.**

A naive generator told *"write a bad pitch because the label is FAILURE"* leaks the answer
into the text — a classifier then reads it off without learning anything. Here the outcome
**emerges**: a salesperson with a hidden skill level talks to a customer with hidden
objections, and a separate evaluator decides, after the fact, whether enough objections
were resolved. The label falls out of arithmetic, not a prompt.

## Pipeline

| Stage | What | Model |
|---|---|---|
| A | Sample all latent state (persona, objections, skill, ...) before any text exists | — (counter-based RNG) |
| B | Generate the conversation one turn at a time | Qwen2.5-3B-Instruct |
| C | Judge each objection against a written criterion | Qwen2.5-7B-Instruct |
| D | `label = SUCCESS if resolved >= threshold else FAILURE` | — (arithmetic) |
| E | Generate the final decision turn | Qwen2.5-7B-Instruct |

The two models run in separate phases; the GPU is cleared between them.

## 0. Environment

Runs on a Colab **L4 GPU**. The pinned versions matter: vLLM 0.6.3 needs this exact
`transformers` / `torch` combination, and the L4 (compute capability 8.9) supports
`bfloat16`. After installing, **restart the runtime once** before running the rest.

In [ ]:
# Check the GPU. Expect: NVIDIA L4, 23034 MiB, 8.9
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# Install vLLM and align the torch / transformers / tokenizers versions it needs.
!pip install -q vllm==0.6.3.post1
!pip install -q "transformers==4.45.2" "tokenizers<0.21"
!pip install -q torchvision==0.19.0 torchaudio==2.4.0 --extra-index-url https://download.pytorch.org/whl/cu121
print("Installed. RESTART THE RUNTIME NOW (Runtime > Restart session), then continue below.")

In [ ]:
# After restarting, verify the environment.
import torch, vllm, numpy, transformers
print("torch        :", torch.__version__)
print("vllm         :", vllm.__version__)
print("transformers :", transformers.__version__)   # expect 4.45.2
print("GPU          :", torch.cuda.get_device_name(0))
print("bf16         :", torch.cuda.is_bf16_supported())

## 1. Stage A — configuration (`personas.py`)

Everything here is sampled **before** any text exists, and nothing is conditioned on an
outcome. Key properties:

- **4 personas**, each owning **4 objections** with a written `resolves_when` criterion
  (2–3 drawn per conversation → 10 possible sets, no 1:1 lookup).
- Demographics and moods are **nested** inside each persona (no incoherent pairings).
- A shared **gym fact sheet** — every salesperson gets the same facts; skill governs how
  well they deploy them, not which they know.
- **Skill levels** written as concrete *behaviour*, not style — this is what makes the
  outcome depend on skill without leaking a style fingerprint.
- **Counter-based RNG**: `sample_spec(i)` is a pure function of `(SEED, i)`, so the data is
  reproducible and a crashed run resumes to identical samples.

In [ ]:
%%writefile personas.py
"""
personas.py — Stage A configuration for the Gym Sales Dataset (V3).

Everything here is sampled BEFORE any text exists.
Nothing here is conditioned on an outcome, because no outcome exists yet.

Key V3 properties enforced by this file:
  * Each persona owns FOUR objections; 2-3 are drawn per conversation.
    (V2 gave each persona ONE fixed trigger -> a 1:1 lookup -> memorisation.)
  * Demographics and moods are NESTED inside the persona.
    (V2 sampled them independently -> "Noam, Anxious First-Timer" + "calculating monthly ROI".)
  * Every objection carries a written `resolves_when` criterion. This is the ONLY
    thing the evaluator (Stage C) is allowed to judge against.
  * Skill levels describe CAPABILITY, never STYLE. See the note in SKILL_LEVELS.
"""

from __future__ import annotations

import random
from typing import Any, Dict, List

# The seed value itself is arbitrary. What matters is the PRIME MULTIPLIER in
# sample_spec(): it spaces different SEED families a million apart, so SEED=42 and
# SEED=43 never produce overlapping streams. Change this only BEFORE generating -
# a new SEED means a completely different dataset.
SEED = 42

# ---------------------------------------------------------------------------
# PERSONAS
# ---------------------------------------------------------------------------
# Each persona:
#   key            : stable id
#   title          : the string written into the dataset
#   background     : who they are (NO tone here - tone comes from `mood`)
#   objections     : exactly 4. Each has id / label / resolves_when.
#   demographics   : compatible with THIS persona only
#   moods          : compatible with THIS persona only
#
# `label` is what the customer actually worries about (fed to the customer agent).
# `resolves_when` is the binary criterion for the evaluator (never shown to either agent).
# ---------------------------------------------------------------------------

PERSONAS: List[Dict[str, Any]] = [
    {
        "key": "yuval",
        "title": "Yuval - Procrastinating CS Student",
        "background": (
            "Stressed and overwhelmed by exams and deadlines. Wants to get in shape but has "
            "started and stopped several times. Worries about time management and about paying "
            "for something he will not actually use."
        ),
        "objections": [
            {
                "id": "no_time",
                "label": "Between exams and assignments my schedule is chaos - I will never actually show up.",
                "resolves_when": (
                    "The salesperson gives SPECIFIC availability (e.g. concrete opening hours such as "
                    "7AM-10PM, seven days a week, or 24/7 access) AND explicitly connects it to an "
                    "irregular or shifting study schedule. Merely saying 'we are open late' or 'you will "
                    "find the time' does NOT resolve it."
                ),
            },
            {
                "id": "wasted_money",
                "label": "I will end up paying for months I never use, like every other subscription I have.",
                "resolves_when": (
                    "The salesperson names a CONCRETE mechanism that ties payment to usage - a membership "
                    "freeze/pause option, a suspension during exam period, a pay-per-entry alternative, or a "
                    "trial/refund window - AND states its terms (how long, at what cost). Generic reassurance "
                    "such as 'you will definitely use it' does NOT resolve it."
                ),
            },
            {
                "id": "no_plan",
                "label": "Even if I go, I would not know what to do inside. I will wander around and waste the hour.",
                "resolves_when": (
                    "The salesperson names a SPECIFIC onboarding artifact - an intro session with a trainer, a "
                    "written starter program, a beginner track, or a plan in the gym app - AND says when or how "
                    "the customer gets it. 'The staff will help you' alone does NOT resolve it."
                ),
            },
            {
                "id": "commute_friction",
                "label": "Getting there and back is half an hour of my day on top of the workout itself.",
                "resolves_when": (
                    "The salesperson gives a CONCRETE answer to the travel cost - access to multiple branches, a "
                    "specific location advantage, parking, or a short-format option (e.g. a 30-minute circuit) that "
                    "makes a shorter visit worthwhile. 'It is not that far' does NOT resolve it."
                ),
            },
        ],
        "demographics": [
            "Male, early 20s - Computer science student juggling a heavy course load",
            "Male, early 20s - Economics student juggling a heavy and stressful course load",
            "Male, 30s - Work-from-home freelancer who desperately needs a reason to leave the house",
            "Female, 20s - Graduate student splitting her week between lab work and teaching",
            "Male, late 20s - Junior developer working long unpredictable sprint hours",
        ],
        "moods": [
            "Hesitant, half-expecting to talk himself out of it",
            "Cynical, has heard every gym pitch before",
            "Distracted, mentally still on tomorrow's deadline",
            "Tired but genuinely hoping this time is different",
            "Guarded, allergic to being sold to",
        ],
    },
    {
        "key": "omer",
        "title": "Omer - Analytical Bargain Hunter",
        "background": (
            "Rational and market-aware. Has already compared competitors and knows their prices. "
            "Evaluates the membership as an investment and expects the numbers to justify themselves."
        ),
        "objections": [
            {
                "id": "price_gap",
                "label": "The gym down the road is about 30% cheaper for what looks like the same thing.",
                "resolves_when": (
                    "The salesperson ADDRESSES THE GAP IN CONCRETE TERMS - naming specific inclusions and what "
                    "they would otherwise cost (e.g. parking, group classes included at no extra charge), or "
                    "quantifying the difference in some way. A quality claim such as 'you get what you pay for' "
                    "or 'our equipment is better' does NOT resolve it."
                ),
            },
            {
                "id": "hidden_fees",
                "label": "What is the registration fee, the card fee, the locker fee? The sticker price is never the real price.",
                "resolves_when": (
                    "The salesperson EXPLICITLY enumerates what is included in the quoted price and either states "
                    "that there are no additional charges, or names the additional charges and their amounts. "
                    "'There are no surprises with us' does NOT resolve it."
                ),
            },
            {
                "id": "contract_lock",
                "label": "Am I locked into a year at this rate, and what happens to the price when it renews?",
                "resolves_when": (
                    "The salesperson states the EXACT commitment term AND what happens at renewal - a price freeze, "
                    "a rolling month-to-month arrangement, or the renewal rate itself. Vague answers such as "
                    "'the price will not jump on you' do NOT resolve it."
                ),
            },
            {
                "id": "peak_congestion",
                "label": "If I pay a premium and then wait twenty minutes for a squat rack, the return on this is zero.",
                "resolves_when": (
                    "The salesperson gives SPECIFIC capacity evidence - equipment counts, a live occupancy indicator "
                    "in the app, documented quiet hours, or a member cap. Inviting him to 'come and see for yourself' "
                    "or saying 'it is never really that crowded' does NOT resolve it."
                ),
            },
        ],
        "demographics": [
            "Male, 40s - High-stress corporate executive",
            "Male, 40s - Blue-collar worker with physical fatigue, weighing every shekel",
            "Female, 30s - Tech professional sitting at a desk ten hours a day",
            "Male, 50s - Small business owner who negotiates for a living",
            "Female, 40s - Financial analyst who priced out three gyms before walking in",
        ],
        "moods": [
            "Analytical, calculating monthly ROI",
            "Assertive, treating this as a negotiation",
            "Skeptical, waiting to catch an inconsistency",
            "Businesslike, short on patience for small talk",
            "Curious but unwilling to be rushed",
        ],
    },
    {
        "key": "ariel",
        "title": "Ariel - Impulsive Commitment-Phobe",
        "background": (
            "High initial motivation but a long history of abandoning projects. Wants to start today, "
            "but the moment the conversation turns to contracts and payment terms, gets defensive."
        ),
        "objections": [
            {
                "id": "contract_trap",
                "label": "A year-long contract? I will quit in six weeks like I always do, and keep paying anyway.",
                "resolves_when": (
                    "The salesperson offers a CONCRETE non-locking arrangement - a rolling monthly membership or "
                    "no-penalty cancellation - AND states the terms (notice period, any fee). 'You will not want to "
                    "quit' or 'we are flexible' does NOT resolve it."
                ),
            },
            {
                "id": "cancellation_maze",
                "label": "Every gym makes cancelling a nightmare - phone calls, registered letters, retention agents.",
                "resolves_when": (
                    "The salesperson names the ACTUAL cancellation mechanism (in the app, a single form at the desk, "
                    "an email) AND the timeframe it takes effect. 'Just talk to us and we will sort it out' does NOT "
                    "resolve it."
                ),
            },
            {
                "id": "boredom",
                "label": "I get bored of the same routine within a month and stop showing up.",
                "resolves_when": (
                    "The salesperson names SPECIFIC varied or rotating offerings - a class schedule with named formats, "
                    "programs that change on a stated cycle, challenges - with concrete examples. 'There is always "
                    "something new' does NOT resolve it."
                ),
            },
            {
                "id": "motivation_dropoff",
                "label": "Honestly, nothing will make me actually keep going after the first excitement wears off.",
                "resolves_when": (
                    "The salesperson names a CONCRETE accountability mechanism - scheduled trainer check-ins, an "
                    "attendance target with a reward, a buddy or group commitment - and describes how it works. "
                    "Motivational encouragement such as 'you just have to want it' does NOT resolve it."
                ),
            },
        ],
        "demographics": [
            "Female, 20s - Social media influencer focused on aesthetics and gym lighting",
            "Female, late 20s - Bride-to-be wanting to tone up before her wedding in four months",
            "Male, 30s - Serial hobbyist with a garage full of abandoned equipment",
            "Female, 30s - Marketing manager who signs up for things impulsively",
            "Male, mid-20s - Recently discharged, chasing the next thing to be excited about",
        ],
        "moods": [
            "Energetic and ready to start today",
            "Enthusiastic, but flinching at the word 'contract'",
            "Impatient, wants the paperwork part to be over",
            "Defensive about being asked to commit",
            "Upbeat but privately certain this will not last",
        ],
    },
    {
        "key": "noam",
        "title": "Noam - Anxious First-Timer",
        "background": (
            "Completely new to fitness, with low body confidence. Has walked past the gym several times "
            "before coming in. Worried less about money than about the experience of being inside the room."
        ),
        "objections": [
            {
                "id": "judgment_fear",
                "label": "Everyone in there is already fit. I will be the only one who does not belong.",
                "resolves_when": (
                    "The salesperson gives a CONCRETE structural answer - a beginner-only zone, specific quiet or "
                    "low-traffic hours, a staff-led introduction, a members' etiquette policy. Emotional reassurance "
                    "such as 'nobody is looking at you' or 'everyone here is friendly' does NOT resolve it."
                ),
            },
            {
                "id": "equipment_confusion",
                "label": "I do not know how to use any of the machines. I will look stupid trying.",
                "resolves_when": (
                    "The salesperson names a SPECIFIC guided pathway - a complimentary intro session, a staff member "
                    "always on the floor, QR-code guides on the machines, a beginners' orientation class - and says "
                    "how to access it. 'You will pick it up quickly' does NOT resolve it."
                ),
            },
            {
                "id": "fitness_level",
                "label": "I am too out of shape to even start. I would not last ten minutes.",
                "resolves_when": (
                    "The salesperson names a SPECIFIC entry-level pathway - a low-intensity beginner class, a graded "
                    "starter program, a shortened first session - rather than encouragement. 'Everybody starts "
                    "somewhere' does NOT resolve it."
                ),
            },
            {
                "id": "changing_room",
                "label": "I do not want to change or shower with other people around.",
                "resolves_when": (
                    "The salesperson names SPECIFIC private facilities - individual changing cubicles, private showers, "
                    "lockers in a separate area - or a concrete alternative (arriving changed, quiet-hour access). "
                    "'It is not a big deal, people mind their own business' does NOT resolve it."
                ),
            },
        ],
        "demographics": [
            "Female, 30s - New parent with very limited free time",
            "Male, early 60s - Recently retired, looking for a morning community and light activity",
            "Female, 50s - Empty nester looking to reinvent her daily routine",
            "Male, 40s - Office worker who has not exercised since school",
            "Female, 40s - Returning to activity after a long medical recovery",
        ],
        "moods": [
            "Soft-spoken, apologising for taking up time",
            "Insecure, bracing to be talked down to",
            "Quietly hopeful but easily discouraged",
            "Embarrassed to be asking basic questions",
            "Nervous, close to walking back out",
        ],
    },
]

PERSONA_BY_KEY: Dict[str, Dict[str, Any]] = {p["key"]: p for p in PERSONAS}


# ---------------------------------------------------------------------------
# OBJECTION KEYWORDS  (measurement only - NEVER shown to any agent)
# ---------------------------------------------------------------------------
# Pilot 2 taught us that "did the customer actually voice his objection?" cannot be
# checked by reading a few conversations by eye. In pilot 2's sample #2 all THREE
# objections went unvoiced while the conversation looked perfectly fluent.
#
# These keyword sets give us a cheap, automatic compliance rate over the whole batch:
#   voiced(obj) = any keyword appears in any customer turn
#
# Crude, but it is a floor, not a verdict - and it turns "I read two conversations and
# they looked wrong" into "objection voicing rate = X%", which is an EDA finding.
# ---------------------------------------------------------------------------

OBJECTION_KEYWORDS: Dict[str, List[str]] = {
    "no_time":            ["time", "schedule", "busy", "exam", "deadline", "hours", "fit it in", "show up", "chaos"],
    "wasted_money":       ["waste", "wasting", "not use", "never use", "pay for", "subscription", "money", "unused"],
    "no_plan":            ["what to do", "no idea", "would not know", "wouldn't know", "plan", "lost", "wander", "structure", "guidance", "where to start"],
    "commute_friction":   ["get there", "getting there", "travel", "commute", "far", "drive", "each way", "trip", "back and forth", "journey", "distance", "minutes away", "getting to"],
    "price_gap":          ["cheaper", "down the road", "competitor", "other gym", "30%", "less expensive", "price", "cost", "elsewhere"],
    "hidden_fees":        ["hidden", "extra", "registration", "joining fee", "card fee", "locker fee", "sticker price", "real price", "additional", "surprise", "fee"],
    "contract_lock":      ["locked", "lock", "year", "12-month", "renew", "renewal", "committed", "commitment", "tied"],
    "peak_congestion":    ["crowded", "busy", "wait", "queue", "squat rack", "packed", "peak", "equipment", "available"],
    "contract_trap":      ["contract", "locked", "quit", "trapped", "stuck", "commit", "year", "tied", "obligation"],
    "cancellation_maze":  ["cancel", "cancelling", "cancellation", "quit", "get out", "leave", "terminate", "opt out"],
    "boredom":            ["bored", "boring", "same routine", "repetitive", "variety", "stop showing", "monotony", "lose interest", "same thing"],
    "motivation_dropoff": ["motivation", "motivated", "keep going", "stick with", "stick to", "give up", "wears off", "accountab", "consistent", "drop off", "keep it up"],
    "judgment_fear":      ["judge", "judged", "look at me", "everyone", "fit people", "belong", "intimidat", "embarrass", "self-conscious", "stare", "watching", "out of place"],
    "equipment_confusion":["machine", "equipment", "how to use", "don't know how", "do not know how", "look stupid", "no idea how", "operate", "unfamiliar"],
    "fitness_level":      ["out of shape", "unfit", "too old", "ten minutes", "not last", "shape", "beginner", "never exercised", "start", "fitness level", "keep up"],
    "changing_room":      ["changing", "change", "shower", "locker room", "private", "undress", "self-conscious", "cubicle"],
}


# ---------------------------------------------------------------------------
# GYM FACT SHEET
# ---------------------------------------------------------------------------
# >>> ADDED IN PILOT 2. THE PILOT-1 FAILURE THAT FORCED IT: <<<
#
# Pilot 1 gave the salesperson a gym name, a price and a skill description - and
# nothing else. But every `resolves_when` above demands a CONCRETE fact: exact opening
# hours, the cancellation mechanism, the number of squat racks. The salesperson did not
# have those facts, so even the EXPERT could only produce "our club offers flexible
# hours and plenty of amenities."
#
# We were asking him to resolve objections without giving him anything to resolve them
# WITH. Nearly every objection would have been scored unresolved, almost every row would
# have been FAILURE, and skill would have predicted nothing. That is the silent failure
# mode of spec 7.2 - a dataset that looks fine and teaches nothing.
#
# CRITICAL: every advisor gets THE SAME sheet, regardless of skill.
# Skill does NOT determine WHICH facts he knows. It determines HOW WELL he deploys them.
# If skill controlled knowledge, "novice" would mean "cannot answer" - a deterministic
# mapping from skill to outcome, which is V2's leakage with extra steps.
# ---------------------------------------------------------------------------

GYM_FACTS: str = """OPENING HOURS: Sun-Thu 06:00-23:00, Fri 06:00-20:00, Sat 08:00-22:00. Open every day.
BRANCHES: 4 branches; one membership covers all of them.
PARKING: free on site, about 120 spaces. Paid lots nearby charge around 25 shekels a visit.

PRICING AND FEES: the quoted monthly price is the full price. No registration fee, no joining
fee, no card fee, no locker fee. Optional towel rental is 8 shekels a visit.
CLASSES: about 35 group classes a week (spinning, yoga, pilates, HIIT, functional, aqua),
all included at no extra charge. Independent studios in the area charge 45-60 shekels a class.

CONTRACT: default is rolling month-to-month. Cancel any time with 30 days notice, either in
the app or with a single form at the front desk. No cancellation penalty, no phone calls
required. An optional 12-month plan is 15 percent cheaper and locks the price for a year.
RENEWAL: the monthly price is fixed for 12 months from the day of joining.
FREEZE: the membership can be frozen for up to 2 months a year at no cost, 7 days notice, in the app.
TRIAL: 7-day free trial pass. First month is refundable within 14 days.
PAY AS YOU GO: single entry 45 shekels; a 10-entry card is 380 shekels.

ONBOARDING: a free 45-minute intro session with a trainer in the first week, which produces a
written 6-week starter program that also lives in the app.
BEGINNER ZONE: a separate 200 sqm beginner floor. Staff are on the floor 06:00-22:00. Every
machine has a QR code with a short how-to video.
BEGINNER CLASS: "First Steps", a low-intensity class for complete beginners, 3 times a week.
A first session can be as short as 20 minutes.
EXPRESS OPTION: a 30-minute guided circuit for people who cannot spare a full hour.

EQUIPMENT AND CROWDING: 6 squat racks, 4 lifting platforms, 12 benches, 30 cardio machines.
The app shows live occupancy. Quiet hours are 06:00-08:00 and 13:00-16:00; the peak is 17:30-20:00.
CHANGING ROOMS: 14 private changing cubicles and individual lockable showers.

VARIETY: the class timetable is rebuilt every 6 weeks and new programs are added quarterly.
ACCOUNTABILITY: a free trainer check-in every 6 weeks; an attendance challenge (12 visits in a
month earns a free month); a referral scheme for training with a friend."""


# ---------------------------------------------------------------------------
# SKILL LEVELS
# ---------------------------------------------------------------------------
# >>> THIS IS THE SINGLE MOST DANGEROUS BLOCK IN THE PROJECT. <<<
#
# The leakage audit (spec 7.1) asks: can a classifier predict the outcome from the
# salesperson's turns BEFORE the customer decides? V2 scored 90.4% because its prompt
# literally said "give a generic pitch and push too hard" for FAILURE rows. The model
# wrote a caricature, and the caricature WAS the label.
#
# So these descriptions specify CAPABILITY (what he knows, what he tracks, what he digs
# into) and NEVER STYLE (pushy / warm / generic / aggressive). A novice here is not a bad
# person giving a bad pitch - he is a decent person who misses things.
#
# If 7.1 comes back > 70%, the fix is HERE: make these three even closer together.
# ---------------------------------------------------------------------------

SKILL_LEVELS: Dict[str, str] = {
    # >>> REWRITTEN AFTER THE FLAT-LABEL FAILURE. <<<
    # The previous version described skill as abstract "capability" and was so soft that
    # novice resolved 0.89 objections and expert 0.92 - a 0.03 gap, statistically nothing.
    # SUCCESS was then 25% at every skill level: the silent failure of spec 7.2.
    #
    # This version gives each level CONCRETE, OBSERVABLE BEHAVIOUR - a procedure to follow,
    # not a personality. Two salespeople can state the same fact; the difference is whether
    # they dig first, quote the exact number, and circle back. That is behaviour, not a
    # style fingerprint, so it should move resolved_count WITHOUT tripping the 7.1 leakage
    # probe (which reads style, not substance).
    "novice": (
        "You are new and you are nervous about losing the sale, so you rush.\n"
        "FOLLOW THESE HABITS:\n"
        "- When the customer raises a worry, give a quick general answer - a reassuring phrase or "
        "a rough idea - rather than digging out the exact number or the precise procedure from what you know.\n"
        "- Answer only the first worry that registers, then steer back toward signing up.\n"
        "- Do NOT ask what is really behind the worry. Take it at face value.\n"
        "- Do NOT go back to a worry once you have moved past it, even if your answer was thin.\n"
        "You are not a bad person - you are just green, and it shows in vague, hurried answers."
    ),
    "average": (
        "You are competent and you know the details, but you are not thorough.\n"
        "FOLLOW THESE HABITS:\n"
        "- Answer the worry the customer actually stated, and usually give a real specific - a number or "
        "a named mechanism - when it is a topic you handle every day.\n"
        "- When the customer raises two worries at once, deal properly with the one you know best and give "
        "the other a lighter, more general answer.\n"
        "- You rarely ask what is behind a worry, and you do not always check whether your answer satisfied them.\n"
        "- If you leave a worry half-answered, you usually do not return to it unless the customer pushes."
    ),
    "expert": (
        "You are seasoned and methodical. You lose sales by leaving doubts alive, so you close them.\n"
        "FOLLOW THESE HABITS:\n"
        "- When a worry comes up, first ask a short question to find out what is really behind it, THEN answer "
        "the real concern.\n"
        "- Always answer with the exact detail - the specific number, the named mechanism, the timeframe - "
        "never with a general reassurance.\n"
        "- Keep track of every worry the customer raises across the whole conversation.\n"
        "- Before you move toward closing, go back and give a concrete answer to any worry you have not yet "
        "fully resolved."
    ),
}


# ---------------------------------------------------------------------------
# CONTEXT VARIABLES
# ---------------------------------------------------------------------------

GYMS: List[str] = [
    "Ironwood Fitness",
    "Pulse Athletic Club",
    "Northgate Strength & Conditioning",
    "The Forge Gym",
    "Vitality Health Club",
    "Basecamp Fitness Studio",
]

PRICES: List[str] = [
    "\u20aa149/month",
    "\u20aa179/month",
    "\u20aa219/month",
    "\u20aa259/month",
    "\u20aa299/month",
]

N_TURNS_OPTIONS: List[int] = [6, 8]  # even: sales-first, customer-last


# ---------------------------------------------------------------------------
# STAGE A - COUNTER-BASED SPEC SAMPLING
# ---------------------------------------------------------------------------
# sample_spec(i) is a PURE FUNCTION of (SEED, i).
#
# Sample #7331 is byte-identical in run 1 or run 5, on any machine, in any batch order,
# whether or not the job crashed and resumed halfway. V2 drew from a global RNG stream,
# so resuming a crashed run silently produced different data under the same seed.
#
# NOTE: nothing below is conditioned on an outcome. There is no outcome yet.
# ---------------------------------------------------------------------------

def sample_spec(i: int) -> Dict[str, Any]:
    """Return the complete hidden initialization for sample `i`."""
    rng = random.Random(SEED * 1_000_003 + i)

    persona = rng.choice(PERSONAS)

    # Demographic and mood are drawn from THIS persona's own compatible lists.
    demographic = rng.choice(persona["demographics"])
    mood = rng.choice(persona["moods"])

    # 2-3 objections drawn from this persona's pool of 4 -> C(4,2)+C(4,3) = 10 sets.
    n_obj = rng.choice([2, 3])
    objections = rng.sample(persona["objections"], n_obj)

    # --- n_turns is COUPLED to n_obj (added in pilot 2) ------------------------
    # The rollout runs turns 0..n_turns-2, so the customer speaks at t = 1, 3, 5...
    #   n_turns=6 -> customer speaks twice   -> room for 2 objections
    #   n_turns=8 -> customer speaks 3 times -> room for 3
    # In pilot 1 these were independent, so a 3-objection customer in a 6-turn
    # conversation physically could not voice his third concern. It was then scored
    # 'unresolved' - a guaranteed FAILURE caused by arithmetic, not by the salesperson.
    n_turns = 8 if n_obj == 3 else rng.choice(N_TURNS_OPTIONS)

    # Threshold in {n-1, n}: either he must resolve all of them, or all but one.
    threshold = rng.choice([n_obj - 1, n_obj])

    # Skill is sampled INDEPENDENTLY of persona, mood, objections and threshold.
    # It is the causal driver of the outcome, so it must not be correlated with the
    # difficulty of the customer.
    skill_level = rng.choice(["novice", "average", "expert"])

    return {
        "sample_id": i,
        "persona_key": persona["key"],
        "persona": persona["title"],
        "persona_background": persona["background"],
        "demographic": demographic,
        "mood": mood,
        "hidden_objection_ids": [o["id"] for o in objections],
        "hidden_objection_labels": [o["label"] for o in objections],
        "hidden_objection_criteria": [o["resolves_when"] for o in objections],
        "persuasion_threshold": threshold,
        "skill_level": skill_level,
        "skill_description": SKILL_LEVELS[skill_level],
        "gym": rng.choice(GYMS),
        "price": rng.choice(PRICES),
        "n_turns": n_turns,
    }


# ---------------------------------------------------------------------------
# COMBINATORICS (spec section 5)
# ---------------------------------------------------------------------------

def combination_space() -> int:
    """Upper bound on distinct hidden specs. Reported in the dataset card."""
    total = 0
    for p in PERSONAS:
        n_dem = len(p["demographics"])
        n_mood = len(p["moods"])
        n_obj_sets = 6 + 4  # C(4,2)=6, C(4,3)=4
        total += n_dem * n_mood * n_obj_sets
    # threshold (2) x skill (3) x gym x price x n_turns
    total *= 2 * len(SKILL_LEVELS) * len(GYMS) * len(PRICES) * len(N_TURNS_OPTIONS)
    return total


if __name__ == "__main__":
    print(f"Combination space: {combination_space():,}\n")

    for i in (0, 1, 4271):
        s = sample_spec(i)
        print(f"--- sample_id={s['sample_id']} ---")
        print(f"  persona    : {s['persona']}")
        print(f"  demographic: {s['demographic']}")
        print(f"  mood       : {s['mood']}")
        print(f"  skill      : {s['skill_level']}")
        print(f"  objections : {s['hidden_objection_ids']}")
        print(f"  threshold  : {s['persuasion_threshold']} / {len(s['hidden_objection_ids'])}")
        print(f"  gym/price  : {s['gym']} @ {s['price']}")
        print(f"  n_turns    : {s['n_turns']}")
        print()

    # Determinism check: the same index must reproduce byte-identically.
    assert sample_spec(4271) == sample_spec(4271)
    print("Determinism check passed: sample_spec(4271) is stable.")


## 2. Stage B — turn-by-turn rollout (`rollout.py`)

The conversation is generated one turn at a time, alternating salesperson and customer.

- The **salesperson** sees his skill, the gym, the price, the transcript — **never** the
  objections or the outcome.
- The **customer** sees his persona and hidden objections, and is told *"you have not made
  up your mind."* Each objection is scheduled to a specific turn so it is actually voiced.
- The **role is assigned by the loop**, not generated, so turns always alternate and every
  turn has exactly `{role, text}` — no structural corruption possible.

Uses `Qwen2.5-3B-Instruct`.

In [ ]:
%%writefile rollout.py
"""
rollout.py — Stage B of the Gym Sales Dataset (V3).

Loads a pre-trained model from Hugging Face and generates the conversations
TURN BY TURN, alternating agents.

  The salesperson sees : his skill level, the gym, the price, the transcript so far.
  The salesperson NEVER sees : the hidden objections, the threshold, any outcome.

  The customer sees : his persona, demographic, mood, his own hidden objections,
                      the transcript so far.
  The customer is told : "You have NOT made up your mind."

Neither agent can leak the label BECAUSE THE LABEL DOES NOT EXIST YET.
It is computed in Stage C/D, after this file has finished.

Rollout produces turns 0 .. n_turns-2, always ENDING ON A SALES TURN (the final pitch).
The customer's decision turn is appended later, in Stage E.

------------------------------------------------------------------------------
STRUCTURAL GUARANTEES — and how they are enforced here
------------------------------------------------------------------------------
  Each turn has exactly {role, text}  -> the dict is BUILT IN PYTHON, not parsed
                                          from model output. Stray keys are impossible.
  role in {sales, customer}           -> assigned by loop parity, never generated.
  Turns alternate sales -> customer   -> loop parity.
  First turn is sales                 -> rollout starts at t=0 (even = sales).
  No "turn N:" prefixes               -> NO JSON EXAMPLE APPEARS IN ANY PROMPT.
                                          (V2 leaked its prompt's JSON example into
                                           7,689 turns. That cannot happen here.)
  Text length 25-350 chars            -> generation budget + post-validation + retry.
------------------------------------------------------------------------------
"""

from __future__ import annotations

import argparse
import gc
import json
import os
import re
from pathlib import Path
from typing import Any, Dict, List

from personas import sample_spec, GYM_FACTS

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------

# PILOT 2 VERDICT: Qwen2.5-1.5B-Instruct is not strong enough for this task.
# Two independent failures, both fatal:
#   (a) INSTRUCTION FOLLOWING - it ignored the per-turn objection instruction. In one
#       conversation all three hidden objections went unvoiced while the dialogue read fluently.
#   (b) HALLUCINATION - it invented "a standard freeze fee, which is 15% of the amount you've
#       paid", CONTRADICTING the fact sheet it had been given ("No cancellation penalty").
#       The 15% was lifted from an unrelated line ("the 12-month plan is 15 percent cheaper").
# Both are documented as EDA findings. The fix is a model with better instruction adherence.
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"   # <-- the pre-trained HF model (Stage B)

MIN_CHARS = 25
MAX_CHARS = 350
MAX_NEW_TOKENS = 110          # ~350 chars of English
MAX_MODEL_LEN = 2048

OUT_DIR = Path("artifacts")
OUT_DIR.mkdir(exist_ok=True)


# ---------------------------------------------------------------------------
# PROMPTS
# ---------------------------------------------------------------------------
# Note what is ABSENT from the salesperson prompt: the objections, the threshold,
# the outcome. He is not writing towards a result. He is doing his job at his
# level of skill, and whatever happens, happens.
#
# Note also what is absent from BOTH prompts: any JSON, any example, any mention
# of turn numbers. The model is asked for one spoken line and nothing else.
# ---------------------------------------------------------------------------

SALES_SYSTEM = """You are a membership advisor at {gym}. Membership is {price}.

{skill_description}

THE FACTS YOU HAVE ABOUT THE GYM:
{facts}

Use only the facts above. Do not invent facts that are not there. If something is genuinely not covered, say so.

You are talking to someone who has walked in to ask about joining. Your job is to understand what they need and to sign them up if the gym is right for them.

Write ONLY your next spoken line, in English. One to three sentences. Speak naturally, as in a real conversation. Do not write your name, do not write stage directions, do not describe what you are doing."""


CUSTOMER_SYSTEM = """You are a person who has walked into {gym} to ask about a membership. It is {price}.

Who you are: {demographic}
{background}

Right now you feel: {mood}

Things that worry you about joining:
{objections}

HOW YOU BEHAVE:
You have NOT made up your mind. Do NOT agree to join, and do NOT walk away. You are still weighing it up.
Do NOT thank the advisor for information. Do NOT say "that sounds good", "great", "sounds perfect", or "thanks for the info".
You are not here to be polite. You are here to get answers to the things that worry you.
If an answer is vague, general, or does not actually address what you asked, say so plainly and press for the specific detail.
If an answer IS specific and does address it, acknowledge that honestly - but stay on your guard about the rest.

Write ONLY your next spoken line, in English. One to three sentences. Speak naturally, as in a real conversation. Do not write your name, do not write stage directions, do not describe what you are doing."""


# The customer speaks at rollout turns 1, 3, 5. Objection k is scheduled for the k-th
# of those. This is NOT leakage: the objections were always his own, and he always saw
# them. We are only guaranteeing that he actually VOICES them.
#
# Pilot 1 left this to chance and the model simply did not raise them - it was too
# agreeable. In conversation #1 all THREE hidden objections went unspoken, so the
# salesperson never had the chance to resolve anything.

def turn_instruction(spec: Dict[str, Any], t: int) -> str:
    """What the customer must do on THIS turn."""
    c = (t - 1) // 2                      # 0-based index among customer turns
    labels = spec["hidden_objection_labels"]

    if c < len(labels):
        return (
            "WHAT YOU MUST DO IN THIS LINE:\n"
            "1. React briefly to what the advisor just said.\n"
            "2. Then you MUST state this worry OUT LOUD, in your own words, concretely and "
            "in full. Do not hint at it. Do not save it for later. Say it now:\n"
            f'   "{labels[c]}"\n'
            "Do not thank him. Do not say it sounds good. Just react, then say your worry."
        )
    return (
        "You have already raised all of your worries. Do not raise a new one. "
        "Look back at what the advisor has told you: if any of your worries got a vague or "
        "hand-waving answer, push on it again and demand something concrete."
    )


def build_messages(spec: Dict[str, Any], dialogue: List[Dict[str, str]], role: str, t: int) -> List[Dict[str, str]]:
    """Assemble the chat messages for ONE agent's next turn."""
    if role == "sales":
        system = SALES_SYSTEM.format(
            gym=spec["gym"],
            price=spec["price"],
            skill_description=spec["skill_description"],
            facts=GYM_FACTS,
        )
    else:
        objections = "\n".join(f"- {o}" for o in spec["hidden_objection_labels"])
        system = CUSTOMER_SYSTEM.format(
            gym=spec["gym"],
            price=spec["price"],
            demographic=spec["demographic"],
            background=spec["persona_background"],
            mood=spec["mood"],
            objections=objections,
        )

    messages = [{"role": "system", "content": system}]

    # Replay the transcript from THIS agent's point of view:
    # his own turns are 'assistant', the other agent's are 'user'.
    for turn in dialogue:
        speaker = "assistant" if turn["role"] == role else "user"
        messages.append({"role": speaker, "content": turn["text"]})

    # The salesperson opens the conversation and so has nothing to reply to.
    if not messages[1:]:
        messages.append({"role": "user", "content": "(the customer has just walked in)"})

    # ---- PILOT 3 FIX: the instruction goes LAST, not in the system prompt --------
    # In pilot 2 the per-turn instruction sat inside a ~250-word system prompt, behind the
    # persona, the mood, the objection list and the behaviour rules. A small model simply
    # did not act on it - what dominated its attention was the immediate conversational
    # context. Moving it to the tail of the final user message puts it in the strongest
    # position in the prompt: the last thing read before generation begins.
    if role == "customer":
        messages[-1]["content"] += "\n\n---\n" + turn_instruction(spec, t)

    return messages


# ---------------------------------------------------------------------------
# CLEANING + VALIDATION
# ---------------------------------------------------------------------------

_PREFIX = re.compile(
    r"^\s*(sales(person)?|advisor|customer|client|me|you)\s*[:\-]\s*",
    re.IGNORECASE,
)
_STAGE = re.compile(r"[\(\[\*][^)\]\*]{0,80}[\)\]\*]")


def clean(text: str) -> str:
    """Strip the artefacts small instruct models habitually add."""
    text = text.strip()
    text = _PREFIX.sub("", text)          # "Salesperson: ..." -> "..."
    text = _STAGE.sub("", text)           # "(smiling) ..."    -> "..."
    text = text.strip().strip('"').strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def truncate_to_sentence(text: str, limit: int = MAX_CHARS) -> str:
    """If over the limit, cut back to the last complete sentence that fits."""
    if len(text) <= limit:
        return text
    cut = text[:limit]
    end = max(cut.rfind(". "), cut.rfind("! "), cut.rfind("? "))
    if end > MIN_CHARS:
        return cut[: end + 1].strip()
    return cut.rsplit(" ", 1)[0].strip()


def is_valid(text: str) -> bool:
    if not (MIN_CHARS <= len(text) <= MAX_CHARS):
        return False
    if re.search(r"[\u0590-\u05FF]", text):   # any Hebrew character -> reject
        return False
    if re.match(r"^\s*turn\s*\d", text, re.IGNORECASE):
        return False
    return True


# ---------------------------------------------------------------------------
# ROLLOUT
# ---------------------------------------------------------------------------

def run(n_samples: int, shard_size: int, resume: bool = True) -> Path:
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    out_path = OUT_DIR / f"rollout_{n_samples}.jsonl"

    done: set[int] = set()
    if resume and out_path.exists():
        with out_path.open() as f:
            for line in f:
                try:
                    done.add(json.loads(line)["sample_id"])
                except Exception:
                    pass
        print(f"[resume] {len(done)} samples already on disk — skipping them.")

    todo = [i for i in range(n_samples) if i not in done]
    if not todo:
        print("[resume] nothing to do.")
        return out_path

    print(f"Loading {MODEL_NAME} from Hugging Face ...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    llm = LLM(
        model=MODEL_NAME,
        dtype="bfloat16",            # L4 (compute 8.9) supports bf16. On a T4 use "float16".
        max_model_len=MAX_MODEL_LEN,
        gpu_memory_utilization=0.85,
        enable_prefix_caching=True,  # the system prompt is re-sent every turn — cache it
        seed=0,
    )

    sp = SamplingParams(
        temperature=0.85,
        top_p=0.92,
        max_tokens=MAX_NEW_TOKENS,
        repetition_penalty=1.08,
        stop=["\n\n", "Customer:", "Salesperson:", "Advisor:"],
    )
    sp_retry = SamplingParams(
        temperature=0.6, top_p=0.9, max_tokens=MAX_NEW_TOKENS, repetition_penalty=1.15,
        stop=["\n\n", "Customer:", "Salesperson:", "Advisor:"],
    )

    fh = out_path.open("a")
    n_retries = 0
    n_dropped = 0

    for shard_start in range(0, len(todo), shard_size):
        shard_ids = todo[shard_start : shard_start + shard_size]
        specs = {i: sample_spec(i) for i in shard_ids}
        dialogues: Dict[int, List[Dict[str, str]]] = {i: [] for i in shard_ids}

        # rollout turns are 0 .. n_turns-2  ->  the deepest is 8-2 = 6
        max_t = max(s["n_turns"] for s in specs.values()) - 2

        print(f"\n=== shard {shard_start // shard_size + 1} "
              f"({len(shard_ids)} samples, ids {shard_ids[0]}..{shard_ids[-1]}) ===")

        for t in range(max_t + 1):
            role = "sales" if t % 2 == 0 else "customer"   # <-- role is NEVER generated

            # A conversation is active at turn t only if it is long enough to have one.
            active = [i for i in shard_ids if t <= specs[i]["n_turns"] - 2]
            if not active:
                continue

            prompts = [
                tokenizer.apply_chat_template(
                    build_messages(specs[i], dialogues[i], role, t),
                    tokenize=False,
                    add_generation_prompt=True,
                )
                for i in active
            ]

            # ONE batched call for every conversation at this depth.
            outs = llm.generate(prompts, sp, use_tqdm=False)

            texts = {i: clean(o.outputs[0].text) for i, o in zip(active, outs)}
            texts = {i: truncate_to_sentence(txt) for i, txt in texts.items()}

            # Retry only the ones that failed validation.
            bad = [i for i in active if not is_valid(texts[i])]
            if bad:
                n_retries += len(bad)
                retry_prompts = [
                    tokenizer.apply_chat_template(
                        build_messages(specs[i], dialogues[i], role, t),
                        tokenize=False,
                        add_generation_prompt=True,
                    )
                    for i in bad
                ]
                retry_outs = llm.generate(retry_prompts, sp_retry, use_tqdm=False)
                for i, o in zip(bad, retry_outs):
                    texts[i] = truncate_to_sentence(clean(o.outputs[0].text))

            for i in active:
                # The dict is constructed HERE. Two keys. Always.
                dialogues[i].append({"role": role, "text": texts[i]})

            print(f"  turn {t} ({role:8s}) — {len(active):4d} conversations"
                  + (f"  [{len(bad)} retried]" if bad else ""))

        # --- write the shard -------------------------------------------------
        for i in shard_ids:
            dlg = dialogues[i]
            s = specs[i]

            # Final integrity gate. A conversation that fails ANY of these is dropped
            # rather than silently written. (V2 wrote 947 rows with the wrong turn count.)
            ok = (
                len(dlg) == s["n_turns"] - 1
                and dlg[0]["role"] == "sales"
                and dlg[-1]["role"] == "sales"
                and all(d["role"] == ("sales" if k % 2 == 0 else "customer")
                        for k, d in enumerate(dlg))
                and all(set(d.keys()) == {"role", "text"} for d in dlg)
                and all(is_valid(d["text"]) for d in dlg)
            )
            if not ok:
                n_dropped += 1
                continue

            fh.write(json.dumps({"sample_id": i, "dialogue": dlg}, ensure_ascii=False) + "\n")

        fh.flush()
        os.fsync(fh.fileno())          # survive a Colab disconnect mid-shard
        print(f"  [checkpoint] wrote through sample {shard_ids[-1]}")

    fh.close()

    print(f"\nDone. retries={n_retries}  dropped={n_dropped}")
    print(f"Wrote: {out_path}")

    del llm
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass

    return out_path


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=200, help="number of samples (pilot: 200)")
    ap.add_argument("--shard", type=int, default=500)
    args = ap.parse_args()
    run(args.n, args.shard)


## 3. Stages C, D, E — evaluate, label, decide (`evaluate.py`)

- **Stage C (evaluate):** a separate, stronger model reads each finished transcript and
  judges, per objection, whether it was resolved — against the written `resolves_when`
  criterion. Using a different model reduces self-evaluation bias.
- **Stage D (label):** `label = SUCCESS if resolved_count >= persuasion_threshold else
  FAILURE`. No model is ever asked to output SUCCESS/FAILURE.
- **Stage E (decision turn):** only after the label exists, one final customer turn
  verbalises the decision — this is a rendering of a result already produced, not leakage.

Produces two files: the **dataset** (dialogue + outcome) and the **metadata** (latent
state — never a feature). Uses `Qwen2.5-7B-Instruct`.

In [ ]:
%%writefile evaluate.py
"""
evaluate.py — Stages C, D, E of the Gym Sales Dataset (V3).

Runs AFTER rollout.py. Reads artifacts/rollout_10000.jsonl (dialogues only, no labels)
and produces the two final files.

  Stage C — THE EVALUATOR
      A separate, stronger model (Qwen2.5-7B-Instruct-AWQ) reads each finished transcript
      plus that conversation's hidden objections, and judges PER OBJECTION whether the
      salesperson genuinely resolved it, against the written `resolves_when` criterion.

      Why a DIFFERENT model than the 1.5B/3B generator: the evaluator is the single point
      of failure for the whole pipeline. If its judgements are noise, `outcome` is noise
      and every downstream number is meaningless. Using a separate, stronger judge reduces
      self-evaluation bias (§9 limitation #2) and is validated by Cohen's kappa in §7.3.

  Stage D — THE LABEL (a rule, not a prompt)
      label = SUCCESS if resolved_count >= persuasion_threshold else FAILURE
      No LLM is ever asked to output SUCCESS or FAILURE. The label falls out of arithmetic.

  Stage E — THE DECISION TURN
      Only AFTER the label is computed, one final customer turn verbalises the decision.
      It becomes dialogue[-1]. This is NOT leakage: the decision was unavailable to both
      agents throughout the rollout; it is a rendering of a result already produced.

OUTPUT (physically separated - see the warning in the spec):
  artifacts/gym_v3_dataset.jsonl    <- dialogue + outcome         (the model's input)
  artifacts/gym_v3_metadata.jsonl   <- skill, resolved flags, ... (NEVER a feature)
"""

from __future__ import annotations

import argparse
import gc
import json
import os
import re
from pathlib import Path
from typing import Any, Dict, List

from personas import sample_spec, PERSONA_BY_KEY

EVAL_MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"   # separate, stronger judge

ART = Path("artifacts")


# ---------------------------------------------------------------------------
# STAGE C — evaluator prompt
# ---------------------------------------------------------------------------
# The judge sees the transcript, ONE objection, and its written criterion. It answers
# with a single structured verdict. Strictness is explicit: vague reassurance is NOT
# resolution. This strictness is what makes `resolves_when` a real test rather than a
# rubber stamp.
# ---------------------------------------------------------------------------

EVAL_SYSTEM = """You are a strict, experienced sales-training assessor. You read a conversation between a gym membership advisor and a potential customer, and you judge whether ONE specific worry the customer had was genuinely resolved by the advisor.

RESOLVED means the advisor gave a CONCRETE, SPECIFIC answer that a reasonable person would accept - a real number, a named mechanism, a clear procedure. Vague reassurance, deflection, changing the subject, or a general promise is NOT resolved.

You judge ONLY against the written criterion you are given. You do not invent your own standard."""

EVAL_USER = """CONVERSATION:
{transcript}

THE CUSTOMER'S WORRY:
"{objection}"

THE CRITERION FOR RESOLVING IT:
{criterion}

Did the advisor resolve this worry, judged strictly against the criterion above?
Answer in exactly this format, nothing else:
VERDICT: RESOLVED or NOT_RESOLVED
REASON: one sentence"""


def transcript_text(dialogue: List[Dict[str, str]]) -> str:
    speaker = {"sales": "ADVISOR", "customer": "CUSTOMER"}
    return "\n".join(f"{speaker[t['role']]}: {t['text']}" for t in dialogue)


_VERDICT = re.compile(r"VERDICT:\s*(RESOLVED|NOT[_\s]?RESOLVED)", re.I)
_REASON = re.compile(r"REASON:\s*(.+)", re.I | re.S)


def parse_verdict(text: str) -> tuple[bool, str]:
    m = _VERDICT.search(text)
    resolved = bool(m) and "not" not in m.group(1).lower()
    r = _REASON.search(text)
    reason = r.group(1).strip().split("\n")[0][:300] if r else text.strip()[:300]
    return resolved, reason


# ---------------------------------------------------------------------------
# STAGE E — decision-turn prompt
# ---------------------------------------------------------------------------
# The customer now KNOWS the outcome (the simulation has decided). He speaks one closing
# line consistent with it. He is NOT told the count or the threshold - only the direction,
# so the wording stays natural.
# ---------------------------------------------------------------------------

DECISION_SYSTEM = """You are the customer in a gym sales conversation. You have now made up your mind. Write ONLY your final spoken line - one or two sentences, in English, natural and in character. Do not explain your reasoning in a list. Do not write your name or stage directions."""

DECISION_USER_SUCCESS = """You have decided to JOIN. The advisor addressed enough of what worried you that you are willing to sign up. Write your closing line agreeing to join - warm but still in your own voice."""

DECISION_USER_FAILURE = """You have decided NOT to join today. Enough of what worried you went unanswered or was brushed off. Write your closing line declining, or saying you need to think about it - polite but clear that you are not signing up now."""


def build_eval_prompts(tokenizer, dialogue, spec) -> List[str]:
    tx = transcript_text(dialogue)
    prompts = []
    for label, crit in zip(spec["hidden_objection_labels"], spec["hidden_objection_criteria"]):
        msgs = [
            {"role": "system", "content": EVAL_SYSTEM},
            {"role": "user", "content": EVAL_USER.format(transcript=tx, objection=label, criterion=crit)},
        ]
        prompts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
    return prompts


def build_decision_prompt(tokenizer, dialogue, outcome) -> str:
    tx = transcript_text(dialogue)
    body = DECISION_USER_SUCCESS if outcome == "SUCCESS" else DECISION_USER_FAILURE
    msgs = [
        {"role": "system", "content": DECISION_SYSTEM},
        {"role": "user", "content": f"CONVERSATION SO FAR:\n{tx}\n\n{body}"},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


# ---------------------------------------------------------------------------
# clean the decision line (reuse the rollout rules)
# ---------------------------------------------------------------------------

_PREFIX = re.compile(r"^\s*(customer|client|me|you)\s*[:\-]\s*", re.I)
_STAGE = re.compile(r"[\(\[\*][^)\]\*]{0,80}[\)\]\*]")


def clean(text: str) -> str:
    text = _PREFIX.sub("", text.strip())
    text = _STAGE.sub("", text)
    text = re.sub(r"\s+", " ", text).strip().strip('"').strip()
    return text[:350]


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------

def run(rollout_path: str, shard_size: int = 1000, resume: bool = True):
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    rows = [json.loads(l) for l in open(rollout_path)]
    print(f"Loaded {len(rows)} conversations from {rollout_path}")

    data_path = ART / "gym_v3_dataset.jsonl"
    meta_path = ART / "gym_v3_metadata.jsonl"

    done: set[int] = set()
    if resume and data_path.exists():
        with data_path.open() as f:
            for line in f:
                try:
                    done.add(json.loads(line)["sample_id"])
                except Exception:
                    pass
        print(f"[resume] {len(done)} already labelled — skipping.")

    todo = [r for r in rows if r["sample_id"] not in done]
    if not todo:
        print("[resume] nothing to do.")
        return data_path, meta_path

    print(f"Loading {EVAL_MODEL} from Hugging Face ...")
    tok = AutoTokenizer.from_pretrained(EVAL_MODEL)
    llm = LLM(
        model=EVAL_MODEL,
        quantization="awq",
        dtype="float16",              # AWQ kernels run in fp16
        max_model_len=2048,
        gpu_memory_utilization=0.90,
        enforce_eager=True,           # safest for AWQ across GPUs
    )
    sp_eval = SamplingParams(temperature=0.0, max_tokens=90)               # deterministic judging
    sp_dec = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=90,
                            stop=["\n\n", "ADVISOR:", "CUSTOMER:"])

    fdata = data_path.open("a")
    fmeta = meta_path.open("a")

    for start in range(0, len(todo), shard_size):
        shard = todo[start:start + shard_size]
        specs = {r["sample_id"]: sample_spec(r["sample_id"]) for r in shard}

        # ---- STAGE C: one prompt per (conversation, objection), all batched ----
        eval_prompts, index = [], []
        for r in shard:
            sid = r["sample_id"]
            ps = build_eval_prompts(tok, r["dialogue"], specs[sid])
            for k, p in enumerate(ps):
                eval_prompts.append(p)
                index.append((sid, k))

        outs = llm.generate(eval_prompts, sp_eval, use_tqdm=False)

        verdicts: Dict[int, Dict[int, tuple]] = {}
        for (sid, k), o in zip(index, outs):
            verdicts.setdefault(sid, {})[k] = parse_verdict(o.outputs[0].text)

        # ---- STAGE D: arithmetic label ----
        labels: Dict[int, str] = {}
        for r in shard:
            sid = r["sample_id"]
            spec = specs[sid]
            flags = [verdicts[sid][k][0] for k in range(len(spec["hidden_objection_ids"]))]
            resolved = sum(flags)
            labels[sid] = "SUCCESS" if resolved >= spec["persuasion_threshold"] else "FAILURE"

        # ---- STAGE E: decision turn (batched) ----
        dec_prompts = [build_decision_prompt(tok, r["dialogue"], labels[r["sample_id"]]) for r in shard]
        dec_outs = llm.generate(dec_prompts, sp_dec, use_tqdm=False)
        decisions = {r["sample_id"]: clean(o.outputs[0].text) for r, o in zip(shard, dec_outs)}

        # ---- write both files ----
        for r in shard:
            sid = r["sample_id"]
            spec = specs[sid]
            dlg = list(r["dialogue"]) + [{"role": "customer", "text": decisions[sid]}]
            flags = [verdicts[sid][k][0] for k in range(len(spec["hidden_objection_ids"]))]
            reasons = [verdicts[sid][k][1] for k in range(len(spec["hidden_objection_ids"]))]

            fdata.write(json.dumps({
                "sample_id": sid,
                "outcome": labels[sid],
                "dialogue": dlg,
                "decision_turn_index": len(dlg) - 1,
                "persona": spec["persona"],
                "demographic": spec["demographic"],
                "mood": spec["mood"],
                "gym": spec["gym"],
                "price": spec["price"],
                "n_turns": spec["n_turns"],
            }, ensure_ascii=False) + "\n")

            fmeta.write(json.dumps({
                "sample_id": sid,
                "skill_level": spec["skill_level"],
                "hidden_objection_ids": spec["hidden_objection_ids"],
                "hidden_objection_labels": spec["hidden_objection_labels"],
                "objections_resolved_flags": flags,
                "objections_resolved_count": int(sum(flags)),
                "persuasion_threshold": spec["persuasion_threshold"],
                "evaluator_reasoning": " | ".join(reasons),
            }, ensure_ascii=False) + "\n")

        fdata.flush(); os.fsync(fdata.fileno())
        fmeta.flush(); os.fsync(fmeta.fileno())
        n_succ = sum(1 for r in shard if labels[r["sample_id"]] == "SUCCESS")
        print(f"  [{start + len(shard):5d}/{len(todo)}] SUCCESS {n_succ}/{len(shard)} "
              f"= {100*n_succ/len(shard):.0f}%")

    fdata.close(); fmeta.close()
    print(f"\nWrote {data_path}\nWrote {meta_path}")

    del llm
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass

    return data_path, meta_path


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--rollout", default="artifacts/rollout_10000.jsonl")
    ap.add_argument("--shard", type=int, default=1000)
    args = ap.parse_args()
    run(args.rollout, args.shard)


## 4. Automated audit (`audit.py`).


In [ ]:
%%writefile audit.py
"""
audit.py — automated quality audit of a rollout.

Pilot 2 looked fine until we read the transcripts. Reading transcripts does not scale to
10,000 rows, so every failure we found by eye becomes a NUMBER here: objection voicing,
customer niceness, premature decisions, and fact hallucinations. Each is also an EDA
finding in its own right — a measurable rate over the whole batch rather than an
impression from reading a handful of conversations.
"""

from __future__ import annotations

import json
import re
from typing import Any, Dict, List

from personas import OBJECTION_KEYWORDS, sample_spec

# ---------------------------------------------------------------------------
# 1. OBJECTION VOICING — the pilot-2 killer
#    If the customer never says it, the salesperson never gets a chance to resolve it,
#    and the evaluator is guaranteed to score it 'unresolved'. The label then measures
#    the customer's compliance, not the salesperson's skill.
# ---------------------------------------------------------------------------

def objection_voiced(dialogue: List[Dict[str, str]], obj_id: str) -> bool:
    text = " ".join(t["text"] for t in dialogue if t["role"] == "customer").lower()
    return any(k in text for k in OBJECTION_KEYWORDS[obj_id])


# ---------------------------------------------------------------------------
# 2. NICENESS — the customer was explicitly forbidden to do this
# ---------------------------------------------------------------------------

NICE = re.compile(
    r"(that sounds (good|great|perfect)|sounds (good|great|perfect)|"
    r"thank you for (the|explaining|all)|thanks for (the|all|explaining)|"
    r"i appreciate (the|you|all)|that clears things up)",
    re.I,
)

# ---------------------------------------------------------------------------
# 3. PREMATURE DECISION — he was told not to decide during the rollout
# ---------------------------------------------------------------------------

DECIDE = re.compile(
    r"(sign me up|where do i sign|i'?ll sign|i'?ll join|i'?ll take it\b|"
    r"i'?ll take the membership|count me in|let'?s do it|i'?m in\b|"
    r"i'?ll pass\b|not interested|no thanks)",
    re.I,
)

# ---------------------------------------------------------------------------
# 4. FACT HALLUCINATION — the salesperson inventing terms that CONTRADICT the sheet.
#    Pilot 2, sample #0: "a standard freeze fee, which is 15% of the amount you've paid."
#    The sheet says there is NO cancellation penalty. He fabricated a charge.
# ---------------------------------------------------------------------------

HALLUCINATION = [
    (re.compile(r"(cancellation|freeze|termination|exit|early[- ]exit)\s+fee", re.I),
     "invented a cancellation/freeze fee (the sheet says there is none)"),
    (re.compile(r"(registration|joining|sign[- ]?up|card|locker)\s+fee\s+(of|is|:)\s*\d", re.I),
     "invented a joining/registration fee (the sheet says there is none)"),
    (re.compile(r"\b(penalty|penalties)\b", re.I),
     "mentioned a penalty (the sheet says no cancellation penalty)"),
    (re.compile(r"\bdeposit\b", re.I),
     "invented a deposit (not in the sheet)"),
]

# A NEGATED mention is not a hallucination - it is the truth.
#   "there are NO cancellation fees"  -> correct, matches the sheet
#   "you'll be charged a freeze fee"  -> fabricated
# Without this guard the detector flags the salesperson for being right.
NEGATED = re.compile(
    r"\b(no|not|never|without|zero|aren'?t|isn'?t|don'?t|doesn'?t|free of|"
    r"nothing in the way of)\b[^.!?]{0,40}$",
    re.I,
)


def is_hallucination(text: str):
    """Return the reason string, or None if the turn is clean."""
    for pat, why in HALLUCINATION:
        m = pat.search(text)
        if not m:
            continue
        # look at the ~40 chars immediately before the match
        before = text[max(0, m.start() - 45): m.start()]
        if NEGATED.search(before):
            continue          # "there are no cancellation fees" -> fine
        return why
    return None


def audit(path: str) -> Dict[str, Any]:
    rows = [json.loads(l) for l in open(path)]
    n = len(rows)

    voiced = miss = 0
    unvoiced_by_id: Dict[str, int] = {}
    nice_turns = nice_convs = 0
    premature = []
    halluc = []
    n_cust = n_sales = 0

    for r in rows:
        s = sample_spec(r["sample_id"])
        dlg = r["dialogue"]

        for oid in s["hidden_objection_ids"]:
            if objection_voiced(dlg, oid):
                voiced += 1
            else:
                miss += 1
                unvoiced_by_id[oid] = unvoiced_by_id.get(oid, 0) + 1

        conv_nice = False
        for t in dlg:
            if t["role"] == "customer":
                n_cust += 1
                if NICE.search(t["text"]):
                    nice_turns += 1
                    conv_nice = True
                if DECIDE.search(t["text"]):
                    premature.append((r["sample_id"], t["text"]))
            else:
                n_sales += 1
                why = is_hallucination(t["text"])
                if why:
                    halluc.append((r["sample_id"], why, t["text"]))
        nice_convs += conv_nice

    tot_obj = voiced + miss

    print(f"=== AUDIT  ({n} conversations) ===\n")

    print(f"1. OBJECTION VOICING      {voiced}/{tot_obj} = {100*voiced/tot_obj:5.1f}%   "
          f"[target > 90%]")
    if unvoiced_by_id:
        worst = sorted(unvoiced_by_id.items(), key=lambda x: -x[1])[:4]
        print("   worst offenders: " + ", ".join(f"{k}({v})" for k, v in worst))

    print(f"\n2. NICENESS VIOLATIONS    {nice_turns}/{n_cust} customer turns = "
          f"{100*nice_turns/n_cust:5.1f}%   [target < 5%]")
    print(f"   conversations affected: {nice_convs}/{n} = {100*nice_convs/n:.1f}%")

    print(f"\n3. PREMATURE DECISIONS    {len(premature)}/{n_cust} = "
          f"{100*len(premature)/n_cust:5.2f}%   [target < 1%]")
    for sid, txt in premature[:3]:
        print(f"   [{sid}] {txt[:80]}")

    print(f"\n4. FACT HALLUCINATIONS    {len(halluc)}/{n_sales} sales turns = "
          f"{100*len(halluc)/n_sales:5.2f}%   [target < 1%]")
    for sid, why, txt in halluc[:3]:
        print(f"   [{sid}] {why}")
        print(f"         \"{txt[:80]}...\"")

    ok = (
        100 * voiced / tot_obj > 90
        and 100 * nice_turns / n_cust < 5
        and 100 * len(premature) / n_cust < 1
        and 100 * len(halluc) / n_sales < 1
    )
    print("\n" + ("PASS - ready to scale to 10,000" if ok
                  else "FAIL - fix before scaling"))

    return {
        "n": n,
        "voicing_rate": voiced / tot_obj,
        "niceness_rate": nice_turns / n_cust,
        "premature_rate": len(premature) / n_cust,
        "hallucination_rate": len(halluc) / n_sales,
        "unvoiced_by_id": unvoiced_by_id,
        "pass": ok,
    }


if __name__ == "__main__":
    import sys
    audit(sys.argv[1] if len(sys.argv) > 1 else "artifacts/rollout_200.jsonl")


## 5. Run Stage B — the rollout

Generates all 12,000 conversations (dialogue only, no labels yet). Shards of 1,000 with a
checkpoint after each, so a Colab disconnect resumes where it left off. ~50 minutes on L4.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import personas, rollout
assert "FOLLOW THESE HABITS" in personas.SKILL_LEVELS["novice"], "old personas.py loaded"

rollout.run(n_samples=12000, shard_size=1000)

**Before scaling, we validated on a 200-sample pilot** and iterated on three problems
found there — the customer not voicing objections, the salesperson lacking facts to resolve
them, and (initially) skill not affecting behaviour. Each fix is reflected in the config
above. Running the audit on a pilot is how those were caught:

In [ ]:
# Optional: audit a pilot before the full run.
import audit
audit.audit("artifacts/rollout_12000.jsonl")   # voicing, politeness, hallucination rates

## 6. Run Stages C/D/E — evaluate and label

**Restart the runtime first** to clear the 3B model from the GPU, then load the 7B
evaluator. Shards of 500 with checkpointing; resumes on disconnect. ~75 minutes on L4.

> Note: the 7B evaluator is loaded **unquantized** (`Qwen2.5-7B-Instruct`, not the AWQ
> build) — the AWQ kernels on this vLLM version intermittently crash with an illegal memory
> access, while the full model runs stably.

In [ ]:
# After restarting: remount, then evaluate.
from google.colab import drive
drive.mount('/content/drive')

import evaluate
evaluate.run("artifacts/rollout_12000.jsonl", shard_size=500)

## 7. Validate the label and save

The single most important check: **SUCCESS must rank novice < average < expert.** Skill was
sampled independently of customer difficulty, so if it still predicts the outcome, that can
only run through the salesperson resolving more objections — a real, learnable signal.

In [ ]:
import json
from collections import defaultdict

data = {json.loads(l)["sample_id"]: json.loads(l) for l in open("artifacts/gym_v3_dataset.jsonl")}
meta = [json.loads(l) for l in open("artifacts/gym_v3_metadata.jsonl")]

by = defaultdict(lambda: [0, 0, []])
for m in meta:
    sid = m["sample_id"]
    by[m["skill_level"]][0] += data[sid]["outcome"] == "SUCCESS"
    by[m["skill_level"]][1] += 1
    by[m["skill_level"]][2].append(m["objections_resolved_count"])

for sk in ["novice", "average", "expert"]:
    s, n, c = by[sk]
    print(f"{sk:8s}: SUCCESS {100*s/n:4.1f}%   avg_resolved {sum(c)/len(c):.3f}   (n={n})")

In [ ]:
# Back up the final files to Drive.
DST = "/content/drive/MyDrive/GSD_IDO"
import os; os.makedirs(DST, exist_ok=True)
!cp artifacts/gym_v3_dataset.jsonl   "$DST"/
!cp artifacts/gym_v3_metadata.jsonl  "$DST"/
print("Saved dataset + metadata to", DST)

The dataset then goes through a separate **EDA notebook** (deduplication, leakage
audit, evaluator validation via Cohen's κ, and failure analysis) before being uploaded to
the Hugging Face Hub. See the EDA notebook and the dataset card for those results.